# Resume saved-attack development — frozen v4

Runs 15 clean resumes and all 60 existing standard attacks (budgets 2, 4, 8, 16), across five methods: 375 method records. No new attacks are generated. No reserved resumes are evaluated. The localization prompt, one-retry policy, geometry and agreement rule are identical to v4. Localization is recomputed on each attacked document; clean boxes and labels are not supplied to the defense.

Upload both resume_development_inputs_v1.zip and resume_complete_inputs_v1.zip to MyDrive/CSCI 566 Project/test/csev_full_colab/resume_colab. Select A100 and run cells in order. Inputs are verified before inference, including image hashes and actual pixel budgets. Results save under resume_saved_attack_development_v1.

This is a development attack comparison, not a final reserved evaluation or direct-adaptive robustness test. Keep all methods and all attacks regardless of success. Share the exported ZIP afterward.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, hashlib, shutil, zipfile, tempfile, os

drive.mount('/content/drive')
RESUME_DRIVE = Path('/content/drive/MyDrive/CSCI 566 Project/test/csev_full_colab/resume_colab')
RESUME_BUNDLE = RESUME_DRIVE / 'resume_development_inputs_v1.zip'
RESUME_ROOT = Path('/content/resume_saved_attack_development_v1')
RESUME_OUT = RESUME_DRIVE / 'resume_saved_attack_development_v1'
assert RESUME_BUNDLE.is_file(), f'Missing bundle: {RESUME_BUNDLE}'
subprocess.run(['nvidia-smi'], check=True)
if not RESUME_ROOT.exists():
    temp = Path(tempfile.mkdtemp(prefix='resume_extract_', dir='/content'))
    local_zip = temp / 'inputs.zip'
    print('Copying inputs from Drive...', flush=True)
    shutil.copy2(RESUME_BUNDLE, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        for entry in z.infolist():
            assert (temp / entry.filename).resolve().is_relative_to(temp.resolve())
        z.extractall(temp)
    local_zip.unlink()
    temp.rename(RESUME_ROOT)
for name, expected in json.loads((RESUME_ROOT/'hashes.json').read_text()).items():
    assert hashlib.sha256((RESUME_ROOT/name).read_bytes()).hexdigest() == expected, name
resume_config = json.loads((RESUME_ROOT/'config.json').read_text())
print('Verified documents:', len(json.loads((RESUME_ROOT/'documents.json').read_text())))
print('Recorded Python:', resume_config['python'])
print('Model settings:', resume_config['model'])


## Install the recorded environment
The notebook kernel stays unchanged. The experiment runs in a separate Python environment using the exported package versions.

In [ ]:
def resume_install(args):
    print(' '.join(map(str,args)), flush=True)
    subprocess.run(list(map(str,args)), check=True)
resume_install([sys.executable, '-m', 'pip', 'install', 'uv'])
RESUME_PYTHON = RESUME_ROOT/'venv/bin/python'
if not RESUME_PYTHON.exists():
    resume_install(['uv','venv','--python',resume_config['python'],'--seed',RESUME_ROOT/'venv'])
v = resume_config['versions']
# This study reproduces the recorded CUDA 12.4 PyTorch environment.
assert v['torch'].endswith('+cu124') and v['torchvision'].endswith('+cu124'), v
resume_install(['uv','pip','install','--python',RESUME_PYTHON,
                '--index-url','https://download.pytorch.org/whl/cu124',
                'torch=='+v['torch'],'torchvision=='+v['torchvision']])
resume_install(['uv','pip','install','--python',RESUME_PYTHON] +
               [k+'=='+value for k,value in v.items() if k not in ('torch','torchvision')])
resume_install([RESUME_PYTHON,'-m','pip','check'])


## Run 75 image cases

Results checkpoint after every case. Rerun the same notebook and output folder to resume with unchanged code and environment; an unfinished case restarts.


In [ ]:
RESUME_COMPLETE = RESUME_DRIVE/'resume_complete_inputs_v1.zip'
assert RESUME_COMPLETE.is_file(), f'Upload the complete input ZIP: {RESUME_COMPLETE}'
local_complete = RESUME_ROOT/'resume_complete_inputs_v1.zip'
print('Copying complete bundle from Drive...', flush=True)
shutil.copy2(RESUME_COMPLETE, local_complete)
RESUME_PREPARER = RESUME_ROOT/'prepare_resume_saved_attacks.py'
RESUME_PREPARER.write_text("import zipfile,json,hashlib,sys\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nroot=Path(sys.argv[1]);bundle=Path(sys.argv[2])\nread=lambda p:json.loads(p.read_text())\nsha=lambda b:hashlib.sha256(b).hexdigest()\nc=read(root/'config.json');clean=read(root/'documents.json');byid={d['record_id']:d for d in clean}\nassert len(byid)==15 and set(byid)==set(c['development_ids'])\nassert not set(byid)&set(c['reserved_ids'])\ncases=[dict(d,case_id='clean_'+str(i),family='clean',epsilon_255=0,attack_target=None) for i,d in enumerate(clean)]\nwith zipfile.ZipFile(bundle) as z:\n manifest=json.loads(z.read('archive_hashes.json'))\n def checked(name):\n  b=z.read(name);assert sha(b)==manifest[name],name;return b\n selected=[]\n for name in sorted(z.namelist()):\n  if '/records/' not in name or not name.endswith('.json'):continue\n  r=json.loads(checked(name))\n  if r['record_id'] in byid:selected.append((name,r))\n keys=[(r['record_id'],r['epsilon_255']) for _,r in selected]\n assert len(keys)==len(set(keys))==60\n assert set(keys)=={(rid,e) for rid in byid for e in (2,4,8,16)}\n for index,(name,r) in enumerate(sorted(selected,key=lambda x:(x[1]['record_id'],x[1]['epsilon_255']))):\n  d=byid[r['record_id']];assert r['status']=='complete'\n  assert r['clean_target']==d['clean_target']\n  assert r['attack_target'] in c['labels'] and r['attack_target']!=d['clean_target']\n  assert r['model_id']==c['model']['model_id'] and r['model_revision']==c['model']['revision']\n  assert r['question']==c['question']\n  for key in ('precision','min_pixels','max_pixels','max_new_tokens'):\n   assert r['processor_settings'][key]==c['model'][key],key\n  assert len(r['images'])==len(d['images'])==r['document_page_count']\n  assert r['attacked_image_sha256']==[im['sha256'] for im in r['images']]\n  images=[]\n  for j,im in enumerate(r['images']):\n   assert im['page']==j\n   stored=im['path'];prefix='outputs/qwen_attack/qwen_resume_validation_v1/'\n   assert stored.count(prefix)==1\n   archive_path=prefix+stored.split(prefix,1)[1]\n   b=checked(archive_path);assert sha(b)==im['sha256']\n   target=root/'saved_attacks'/f'{index:03d}'/f'page_{j:02d}.png';target.parent.mkdir(parents=True,exist_ok=True)\n   target.write_bytes(b)\n   with Image.open(target) as attacked,Image.open(root/d['images'][j]['path']) as original:\n    aa=np.asarray(attacked.convert('RGB'),dtype=np.int16);cc=np.asarray(original.convert('RGB'),dtype=np.int16)\n    assert aa.shape==cc.shape,(r['record_id'],j)\n    assert int(np.abs(aa-cc).max())<=r['epsilon_255'],(r['record_id'],j,'pixel budget exceeded')\n   images.append({'path':str(target.relative_to(root)),'sha256':im['sha256']})\n  cases.append({'record_id':r['record_id'],'clean_target':d['clean_target'],'images':images,\n    'case_id':r['experiment_key'],'family':'standard','epsilon_255':r['epsilon_255'],'attack_target':r['attack_target'],\n    'source_record':name,'source_record_sha256':manifest[name],'historical_targeted_success':r['targeted_success_reloaded']})\n(root/'attack_cases.json').write_text(json.dumps(cases,indent=2))\nprint('Verified 15 clean + 60 saved attacks, all page hashes, page order, model settings, and pixel bounds. Reserved documents excluded.')\n")
subprocess.run([str(RESUME_PYTHON),'-u',str(RESUME_PREPARER),str(RESUME_ROOT),str(local_complete)],check=True)


In [ ]:
RESUME_RUNNER = RESUME_ROOT / "resume_saved_attack_development.py"
RESUME_RUNNER.write_text('"""Clean development only: multi-page contextual education views."""\nimport argparse,hashlib,json,math,sys,time,os,platform,importlib.metadata,csv\nfrom pathlib import Path\nLOC=\'Find the education and qualifications sections on this resume page. Locate whole sections, including their headings, degree or diploma names, institutions, dates and any continuation lines. Include stated qualifications even if the page does not explicitly say completed or awarded. Include education entries outside a heading too. Do not infer qualifications from job titles, work experience or skills. Group nearby lines in the same section into one box rather than returning a box per word or line. Return only JSON in this exact format: {"boxes": [[left, top, right, bottom]]}. Use pixel coordinates on the displayed image of width {w} and height {h}. Return {"boxes": []} only if no education or qualification entries are visible. Return at most four section boxes. Do not include explanations or transcribed text.\'\nPOLICY={\'version\':\'resume_development_retry_v4\',\'localization_attempts\':2,\'minimum_height\':.25,\'margin\':.05,\'expansion\':.05,\'max_height\':.70,\'expanded_max_height\':.85,\'max_regions_per_page\':4,\'max_raw_boxes\':16,\'max_crops_per_document\':8,\'localization_tokens\':768,\'fixed_band\':[.2,.8]}\ndef read(p):return json.loads(Path(p).read_text())\ndef sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef write(p,x):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(\'.tmp\');t.write_text(json.dumps(x,indent=2,allow_nan=False));t.replace(p)\ndef parse(raw,processed,original):\n import re\n s=raw.strip()\n # Accept one fenced JSON response, even when preceded by explanation.\n fences=re.findall(r"```(?:json)?\\s*(.*?)```",s,flags=re.S|re.I)\n if fences:\n  if len(fences)!=1:raise ValueError(\'Ambiguous multiple JSON blocks\')\n  s=fences[0].strip()\n empty_phrases={\n  \'there are none.\',\n  \'there are no regions indicating completed education qualifications or awarded degrees on this resume page.\',\n  \'there are no regions stating completed education qualifications or awarded degrees on this resume page.\'}\n if s.lower() in empty_phrases:return []\n # Only repair trailing commas outside strings; do not extract arbitrary numbers.\n cleaned=[];quoted=False;escaped=False\n for i,ch in enumerate(s):\n  if quoted:\n   cleaned.append(ch)\n   if escaped:escaped=False\n   elif ch==\'\\\\\':escaped=True\n   elif ch==\'"\':quoted=False\n  else:\n   if ch==\'"\':quoted=True\n   if ch==\',\' and s[i+1:].lstrip().startswith((\']\', \'}\')):continue\n   cleaned.append(ch)\n obj=json.loads(\'\'.join(cleaned))\n def unpack(x):\n  if isinstance(x,dict):\n   schemas=[k for k in (\'boxes\',\'bbox_2d\') if k in x]\n   named=all(k in x for k in (\'left\',\'top\',\'right\',\'bottom\'))\n   if len(schemas)+int(named)!=1:raise ValueError(\'Unknown or ambiguous box schema\')\n   return unpack(x[schemas[0]]) if schemas else [[x[k] for k in (\'left\',\'top\',\'right\',\'bottom\')]]\n  if isinstance(x,list):\n   if len(x)==4 and all(isinstance(v,(int,float)) and not isinstance(v,bool) for v in x):return [x]\n   out=[]\n   for item in x:out.extend(unpack(item))\n   return out\n  raise ValueError(\'Invalid box representation\')\n vals=unpack(obj)\n # Bound raw output, but merge line boxes before limiting final regions.\n if len(vals)>POLICY[\'max_raw_boxes\']:raise ValueError(\'Too many raw boxes\')\n pw,ph=processed;w,h=original;out=[]\n for v in vals:\n  if len(v)!=4 or any(isinstance(x,bool) or not isinstance(x,(int,float)) or not math.isfinite(x) for x in v):raise ValueError(\'Invalid coordinates\')\n  l,t,r,b=v\n  if l>=r or t>=b:raise ValueError(\'Box endpoints inverted or zero area\')\n  if not (0<=l and r<=pw and 0<=t and b<=ph):raise ValueError(\'Box outside processed image\')\n  top,bottom=round(t*h/ph),round(b*h/ph)\n  if top>=bottom:raise ValueError(\'Box collapses when mapped to original image\')\n  out.append((0,top,w,bottom))\n return out\n\ndef views(boxes,size):\n w,h=size;intervals=[]\n for _,t,_,b in boxes:\n  height=max(round(.25*h),b-t+2*round(.05*h))\n  if height>.70*h:return None\n  top=max(0,min(h-height,round((t+b-height)/2)));intervals.append((top,top+height))\n merged=[]\n for t,b in sorted(intervals):\n  if merged and t<=merged[-1][1]:merged[-1]=(merged[-1][0],max(b,merged[-1][1]))\n  else:merged.append((t,b))\n if len(merged)>POLICY[\'max_regions_per_page\']:return None\n pairs=[]\n for t,b in merged:\n  t2=max(0,t-round(.05*h));b2=min(h,b+round(.05*h))\n  if b-t>.70*h or b2-t2>.85*h:return None\n  pairs.append(((0,t,w,b),(0,t2,w,b2)))\n return pairs\n\ndef localize(model,im,rs,grid):\n prompt=LOC.replace(\'{w}\',str(rs[0])).replace(\'{h}\',str(rs[1]))\n attempts=[];previous_tokens=model.max_new_tokens\n try:\n  model.max_new_tokens=POLICY[\'localization_tokens\']\n  for attempt in range(POLICY[\'localization_attempts\']):\n   query=prompt\n   if attempt:\n    query+=\' Your previous localization output was invalid. Inspect the page again and return fresh section boxes. Every box must satisfy 0 <= left < right <= \'+str(rs[0])+\' and 0 <= top < bottom <= \'+str(rs[1])+\'. Use positive-area boxes, group nearby education lines, and output only the requested JSON. Do not guess any qualification.\'\n   raw=model.predict([im],query)\n   assert model.last_input_info[\'image_grid_thw\']==grid\n   try:\n    boxes=parse(raw,rs,im.size);pairs=views(boxes,im.size)\n    if pairs is None:raise ValueError(\'Merged regions exceed size or count limits\')\n    attempts.append({\'raw\':raw,\'status\':\'valid\'})\n    return {\'raw\':raw,\'processed_size\':rs,\'boxes\':boxes,\'pairs\':pairs,\'status\':\'ok\' if pairs else \'no_education\',\'attempts\':attempts}\n   except (ValueError,KeyError,TypeError) as exc:\n    attempts.append({\'raw\':raw,\'status\':\'invalid\',\'error\':str(exc)})\n  return {\'raw\':attempts[-1][\'raw\'],\'processed_size\':rs,\'status\':\'invalid\',\'error\':attempts[-1][\'error\'],\'attempts\':attempts}\n finally:model.max_new_tokens=previous_tokens\n\ndef main():\n p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--out\',required=True);a=p.parse_args();root=Path(a.root);out=Path(a.out)\n hashes=read(root/\'hashes.json\')\n for name,h in hashes.items():assert sha(root/name)==h,name\n c=read(root/\'config.json\');docs=read(root/\'attack_cases.json\')\n assert len(docs)==75 and {d[\'record_id\'] for d in docs}==set(c[\'development_ids\'])\n assert not set(c[\'development_ids\'])&set(c[\'reserved_ids\'])\n for k,v in c[\'versions\'].items():assert importlib.metadata.version(k)==v,(k,v)\n assert platform.python_version()==c[\'python\']\n import torch\n from PIL import Image\n sys.path.insert(0,str(root));from qwen_adapter import QwenVL\n assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()\n contract={\'inputs\':sha(root/\'hashes.json\'),\'attack_cases\':sha(root/\'attack_cases.json\'),\'frozen_v4_source_sha256\':\'19f651e1c296e8de53795b3a8a35e189541b246c4022a7a3fcc9121e1a401ca2\',\'script\':sha(__file__),\'policy\':POLICY,\'prompt\':LOC,\'gpu\':torch.cuda.get_device_name(),\'versions\':c[\'versions\'],\'cuda\':torch.version.cuda}\n out.mkdir(parents=True,exist_ok=True)\n if (out/\'contract.json\').exists():assert read(out/\'contract.json\')==contract,\'Changed inputs/environment/code: use a new output directory.\'\n else:write(out/\'contract.json\',contract)\n os.environ[\'HF_HOME\']=str(root/\'hf_cache\');torch.manual_seed(566)\n model=QwenVL(device=torch.device(\'cuda\'),**c[\'model\']);norm=lambda x:x.strip().lower() if isinstance(x,str) and x.strip().lower() in c[\'labels\'] else None\n results=[]\n try:\n  for i,d in enumerate(docs):\n   dest=out/\'cases\'/f\'{i:02d}.json\'\n   if dest.exists():results.append(read(dest));print(f\'RESUME {i+1}/{len(docs)}\',flush=True);continue\n   for im in d[\'images\']:assert sha(root/im[\'path\'])==im[\'sha256\'],im[\'path\']\n   pages=[Image.open(root/x[\'path\']).convert(\'RGB\') for x in d[\'images\']];start=time.monotonic()\n   fullraw=model.predict(pages,c[\'question\']);full=norm(fullraw)\n   fixedraw=model.predict([im.crop((0,round(.2*im.height),im.width,round(.8*im.height))) for im in pages],c[\'question\'])\n   regions=[];bands=[];expanded=[];valid=True\n   for j,im in enumerate(pages):\n    # Obtain the exact processor grid, then ask explicitly for that coordinate system.\n    proc=model.processor.image_processor(images=[im],return_tensors=\'pt\');grid=proc[\'image_grid_thw\'].tolist();assert len(grid)==1\n    _,gh,gw=grid[0];patch=model.processor.image_processor.patch_size;rs=(gw*patch,gh*patch);del proc\n    region=localize(model,im,rs,grid);region[\'page\']=j;regions.append(region)\n    if region[\'status\']==\'invalid\':\n     valid=False;continue\n    for k,(b,e) in enumerate(region[\'pairs\']):\n     bands.append(im.crop(b));expanded.append(im.crop(e))\n     target=out/\'crops\'/f\'{i:02d}_{j:02d}_{k:02d}.png\';target.parent.mkdir(exist_ok=True);im.crop(b).save(target)\n     im.crop(e).save(target.with_name(target.stem+\'_expanded.png\'))\n   raw1=raw2=None\n   if valid and 0<len(bands)<=POLICY[\'max_crops_per_document\']:\n    raw1=model.predict(bands,c[\'question\']);raw2=model.predict(expanded,c[\'question\'])\n   crop,wide=norm(raw1),norm(raw2)\n   preds={\'none\':full,\'fixed_middle\':norm(fixedraw),\'localized_only\':crop,\'full_region_agreement\':full if full is not None and full==crop else None,\'contextual_recovery\':crop if crop is not None and crop==wide else None}\n   row={\'case_id\':d[\'case_id\'],\'family\':d[\'family\'],\'epsilon_255\':d[\'epsilon_255\'],\'attack_target\':d[\'attack_target\'],\'record_id\':d[\'record_id\'],\'clean_target\':d[\'clean_target\'],\'page_count\':len(pages),\'regions\':regions,\'crop_count\':len(bands),\'localization_valid\':valid,\n        \'raw_answers\':{\'full\':fullraw,\'fixed\':fixedraw,\'localized\':raw1,\'expanded\':raw2},\'methods\':{k:{\'prediction\':v,\'correct\':v is not None and v==d[\'clean_target\'],\'accepted\':v is not None,\'targeted\':d[\'attack_target\'] is not None and v==d[\'attack_target\']} for k,v in preds.items()},\'elapsed_seconds\':time.monotonic()-start}\n   write(dest,row);results.append(row);print(f\'{i+1}/{len(docs)} {d["case_id"]} pages={len(pages)} regions={len(bands)} \'+str(preds),flush=True)\n  summary=[]\n  for family,epsilon in [(\'clean\',0)]+[(\'standard\',e) for e in (2,4,8,16)]+[(\'standard\',\'all\')]:\n   group=[r for r in results if r[\'family\']==family and (epsilon==\'all\' or r[\'epsilon_255\']==epsilon)]\n   for method in results[0][\'methods\']:\n    correct=sum(r[\'methods\'][method][\'correct\'] for r in group);accepted=sum(r[\'methods\'][method][\'accepted\'] for r in group)\n    targeted=sum(r[\'methods\'][method][\'targeted\'] for r in group);n=len(group)\n    summary.append({\'family\':family,\'epsilon_255\':epsilon,\'method\':method,\'n\':n,\'correct\':correct,\'wrong\':accepted-correct,\'rejected\':n-accepted,\'targeted\':targeted,\'correct_rate\':correct/n,\'coverage\':accepted/n,\'accepted_error_rate\':(accepted-correct)/n,\'target_rate\':targeted/n})\n  write(out/\'summary.json\',{\'complete\':len(results)==75,\'cases\':len(results),\'method_records\':len(results)*5,\'groups\':summary,\'note\':\'Same 15 development documents. 60 standard saved attacks were not optimized against recovery. Repeated variants are not independent documents. Reserved resumes excluded.\'})\n  with (out/\'summary.csv\').open(\'w\',newline=\'\') as f:\n   w=csv.DictWriter(f,fieldnames=list(summary[0]));w.writeheader();w.writerows(summary)\n  print(json.dumps(summary,indent=2))\n finally:model.unload()\nif __name__==\'__main__\':main()\n')
print("Frozen v4 rule; saved-attack runner ready")


In [ ]:
from datetime import datetime
RESUME_OUT.mkdir(parents=True, exist_ok=True)
resume_log = RESUME_OUT / ('run_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'.log')
resume_env = os.environ.copy()
resume_env['MPLBACKEND'] = 'Agg'
with resume_log.open('w') as log:
    process = subprocess.Popen([str(RESUME_PYTHON),'-u',str(RESUME_RUNNER),
        '--root',str(RESUME_ROOT),'--out',str(RESUME_OUT)],
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=resume_env)
    try:
        for line in process.stdout:
            print(line,end='');log.write(line);log.flush()
        status = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired: process.kill();process.wait()
        raise
assert status == 0, f'Run failed; share the log: {resume_log}'


## Review and export
Share this ZIP before moving to reserved resumes or attacks. It includes answers, predicted regions, diagnostic crops and logs. Keep the original inputs in Drive.

In [ ]:
import csv
summary = json.loads((RESUME_OUT/'summary.json').read_text())
print(json.dumps(summary, indent=2))
archive = RESUME_DRIVE/'resume_saved_attack_development_v1.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RESUME_OUT.rglob('*')):
        if f.is_file() and f.suffix in ('.json','.csv','.png','.log'):
            z.write(f, f.relative_to(RESUME_OUT))
print('Share this file:', archive)
from google.colab import files
files.download(str(archive))
